# https://SenatorovAI.com/




## Что мы сделаем

В этом notebook мы построим простой Machine Learning pipeline для предсказания цены дома (`SalePrice`).

Логика близка к статье:

1. Скачать и загрузить Excel dataset.
2. Посмотреть структуру данных.
3. Сделать небольшой EDA.
4. Очистить данные.
5. Закодировать категориальные признаки через One-Hot Encoding.
6. Разделить данные на train и validation в пропорции 80/20.
7. Обучить SVR, RandomForestRegressor и LinearRegression.
8. Сравнить модели по MAPE, MAE и RMSE.

Важно: это учебный baseline. Во втором notebook будет более правильная data science схема с отдельными train, validation и test.

## 1. Импорт библиотек

Библиотеки - это готовые инструменты Python.

- `pandas` нужен для таблиц.
- `numpy` нужен для чисел и массивов.
- `matplotlib` и `seaborn` нужны для графиков.
- `sklearn` нужен для моделей машинного обучения и метрик.
- `pathlib` помогает удобно работать с путями к файлам.

In [ ]:
# pathlib.Path дает удобный объект для путей к файлам и папкам.
from pathlib import Path

# urllib.request.urlopen умеет скачивать файл по URL.
from urllib.request import urlopen, Request

# ssl нужен, чтобы явно управлять проверкой сертификатов при скачивании учебного файла.
import ssl

# pandas - главная библиотека для работы с табличными данными.
import pandas as pd

# numpy помогает с математическими операциями и массивами.
import numpy as np

# matplotlib.pyplot нужен для базовых графиков.
import matplotlib.pyplot as plt

# seaborn строит красивые статистические графики поверх matplotlib.
import seaborn as sns

# train_test_split делит таблицу на обучающую и проверочную части.
from sklearn.model_selection import train_test_split

# OneHotEncoder переводит текстовые категории в числовые dummy-признаки.
from sklearn.preprocessing import OneHotEncoder

# SVR - Support Vector Regression, регрессионная версия SVM.
from sklearn.svm import SVR

# RandomForestRegressor - ансамбль деревьев решений для регрессии.
from sklearn.ensemble import RandomForestRegressor

# LinearRegression - простая линейная регрессия.
from sklearn.linear_model import LinearRegression

# Метрики регрессии: чем меньше ошибка, тем лучше модель.
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, mean_squared_error, r2_score

# Настраиваем стиль графиков, чтобы они выглядели аккуратно.
sns.set_theme(style="whitegrid")

# Фиксируем random_state, чтобы результаты повторялись при новом запуске.
RANDOM_STATE = 42

## 2. Скачивание и загрузка dataset

Dataset лежит в Excel-файле `HousePricePrediction.xlsx`.

Если файл уже есть локально, мы его не скачиваем повторно. Если файла нет, notebook скачает его из GeeksforGeeks media storage.

Целевая переменная называется `SalePrice`: это цена дома, которую мы хотим предсказывать.

In [ ]:
# Прямая ссылка на Excel dataset из статьи GeeksforGeeks.
DATA_URL = "https://media.geeksforgeeks.org/wp-content/uploads/20260116164015462273/HousePricePrediction.xlsx"

# Папка data будет лежать внутри репозитория рядом с notebooks и help.
DATA_DIR = Path("../data/house_price")

# Создаем папку data/house_price, если ее еще нет.
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Полный путь к локальному Excel-файлу.
DATA_PATH = DATA_DIR / "HousePricePrediction.xlsx"

# Если Excel-файла еще нет, скачиваем его.
if not DATA_PATH.exists():
    # Создаем HTTP-запрос с User-Agent, чтобы сервер понял, что это обычный учебный download.
    request = Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    
    # Создаем SSL context без строгой проверки сертификата, потому что на некоторых локальных машинах
    # учебный download может падать из-за корпоративного или системного certificate chain.
    ssl_context = ssl._create_unverified_context()
    
    # Открываем URL и читаем байты Excel-файла.
    with urlopen(request, context=ssl_context, timeout=30) as response:
        file_bytes = response.read()
    
    # Записываем скачанные байты в локальный .xlsx файл.
    DATA_PATH.write_bytes(file_bytes)

# read_excel читает Excel-файл в pandas DataFrame.
dataset = pd.read_excel(DATA_PATH)

# Показываем первые пять строк, чтобы понять, как выглядит таблица.
dataset.head()

## 3. Первый взгляд на данные

Перед обучением модели нужно понять:

- сколько строк и колонок в таблице;
- какие есть типы данных;
- есть ли пропуски;
- какие колонки являются числовыми, а какие категориальными.

In [ ]:
# shape возвращает размер таблицы: количество строк и количество колонок.
print("Dataset shape:", dataset.shape)

# info показывает тип каждой колонки и количество непустых значений.
dataset.info()

In [ ]:
# select_dtypes(include=['object']) выбирает текстовые/категориальные колонки.
object_cols = dataset.select_dtypes(include=["object"]).columns.tolist()

# select_dtypes(include=['int64']) выбирает целочисленные колонки.
integer_cols = dataset.select_dtypes(include=["int64"]).columns.tolist()

# select_dtypes(include=['float64']) выбирает колонки с дробными числами.
float_cols = dataset.select_dtypes(include=["float64"]).columns.tolist()

# Печатаем количество колонок каждого типа.
print("Categorical variables:", len(object_cols), object_cols)
print("Integer variables:", len(integer_cols), integer_cols)
print("Float variables:", len(float_cols), float_cols)

## 4. EDA: числовые признаки и корреляции

Корреляция показывает, насколько два числовых признака связаны друг с другом.

- Значение около `1` означает сильную положительную связь.
- Значение около `-1` означает сильную отрицательную связь.
- Значение около `0` означает слабую линейную связь.

Для цены дома полезно посмотреть, какие числовые признаки сильнее связаны с `SalePrice`.

In [ ]:
# Выбираем только числовые колонки, потому что corr() работает с числами.
numerical_dataset = dataset.select_dtypes(include=["int64", "float64"])

# Создаем область графика размером 12 на 6 дюймов.
plt.figure(figsize=(12, 6))

# Строим heatmap корреляций между числовыми признаками.
sns.heatmap(
    numerical_dataset.corr(),  # матрица корреляций
    cmap="BrBG",               # цветовая схема
    fmt=".2f",                 # показываем числа с двумя знаками после точки
    linewidths=2,              # расстояние между ячейками
    annot=True                 # печатаем числа внутри ячеек
)

# Добавляем заголовок графика.
plt.title("Correlation heatmap of numerical features")

# tight_layout уменьшает риск наложения подписей.
plt.tight_layout()

# Показываем график в notebook.
plt.show()

## 5. EDA: категориальные признаки

Категориальные признаки - это текстовые колонки вроде типа здания или zoning class.

Модели sklearn не умеют напрямую работать со строками, поэтому позже мы превратим такие колонки в числа через One-Hot Encoding.

Сначала посмотрим, сколько уникальных значений есть в каждой категориальной колонке.

In [ ]:
# Для каждой категориальной колонки считаем количество уникальных значений.
unique_values = [dataset[col].nunique(dropna=False) for col in object_cols]

# Создаем область графика.
plt.figure(figsize=(10, 5))

# Строим barplot: по x названия колонок, по y количество уникальных значений.
sns.barplot(x=object_cols, y=unique_values)

# Поворачиваем подписи, чтобы они не накладывались друг на друга.
plt.xticks(rotation=45)

# Добавляем заголовок.
plt.title("Number of unique values in categorical features")

# Подписываем ось y.
plt.ylabel("Unique values")

# Делаем layout аккуратнее.
plt.tight_layout()

# Показываем график.
plt.show()

In [ ]:
# Создаем несколько маленьких графиков для распределения категорий.
fig, axes = plt.subplots(nrows=len(object_cols), ncols=1, figsize=(10, 4 * len(object_cols)))

# Если категориальная колонка одна, axes не будет списком; приводим к списку для универсальности.
if len(object_cols) == 1:
    axes = [axes]

# Проходим по каждой категориальной колонке и соответствующей оси графика.
for ax, col in zip(axes, object_cols):
    # value_counts считает, сколько раз встречается каждая категория.
    counts = dataset[col].value_counts(dropna=False)
    
    # Строим barplot для текущей колонки.
    sns.barplot(x=counts.index.astype(str), y=counts.values, ax=ax)
    
    # Даем графику понятный заголовок.
    ax.set_title(f"Distribution of {col}")
    
    # Поворачиваем подписи категорий.
    ax.tick_params(axis="x", rotation=45)

# Делаем layout аккуратнее.
plt.tight_layout()

# Показываем все графики.
plt.show()

## 6. Очистка данных

В учебной статье используется простой подход:

1. Удалить `Id`, потому что это просто номер записи, а не полезный признак.
2. Заполнить пропуски в `SalePrice` средним значением.
3. Удалить оставшиеся строки с пропусками.

Для production-проекта лучше использовать более аккуратный preprocessing pipeline, но здесь мы повторяем baseline-логику для обучения.

In [ ]:
# copy создает копию таблицы, чтобы не менять исходный dataset напрямую.
clean_dataset = dataset.copy()

# Удаляем Id, если такая колонка есть в таблице.
if "Id" in clean_dataset.columns:
    # axis=1 означает, что удаляем колонку, а не строку.
    clean_dataset = clean_dataset.drop("Id", axis=1)

# Заполняем пропуски в целевой колонке SalePrice средним значением SalePrice.
clean_dataset["SalePrice"] = clean_dataset["SalePrice"].fillna(clean_dataset["SalePrice"].mean())

# Удаляем строки, где остались любые пропуски в других колонках.
clean_dataset = clean_dataset.dropna()

# Проверяем, что пропусков больше нет.
clean_dataset.isnull().sum()

## 7. One-Hot Encoding

Модели машинного обучения ожидают числовую матрицу признаков.

Категориальная колонка `BldgType` может содержать значения вроде:

```text
1Fam, 2fmCon, Duplex, TwnhsE
```

One-Hot Encoding создает отдельную колонку для каждой категории:

```text
BldgType_1Fam, BldgType_2fmCon, BldgType_Duplex, ...
```

Значение в такой колонке будет `1`, если объект относится к категории, и `0`, если нет.

In [ ]:
# Находим все колонки с типом object после очистки данных.
object_cols = clean_dataset.select_dtypes(include=["object"]).columns.tolist()

# Печатаем список категориальных признаков.
print("Categorical columns:", object_cols)

# Создаем OneHotEncoder.
OH_encoder = OneHotEncoder(
    sparse_output=False,    # хотим обычный dense array, а не sparse matrix
    handle_unknown="ignore" # если появится неизвестная категория, не падать с ошибкой
)

# fit_transform сначала изучает категории, потом превращает их в числовую матрицу.
OH_array = OH_encoder.fit_transform(clean_dataset[object_cols])

# Превращаем numpy array обратно в DataFrame с понятными именами колонок.
OH_cols = pd.DataFrame(
    OH_array,
    index=clean_dataset.index,
    columns=OH_encoder.get_feature_names_out(object_cols)
)

# Удаляем исходные текстовые колонки из таблицы.
numeric_part = clean_dataset.drop(object_cols, axis=1)

# Объединяем числовые колонки и one-hot encoded колонки.
df_final = pd.concat([numeric_part, OH_cols], axis=1)

# Показываем размер финальной таблицы после encoding.
print("Final encoded shape:", df_final.shape)

# Показываем первые строки финальной таблицы.
df_final.head()

## 8. Разделение на X и y, train и validation

В supervised learning есть:

- `X` - признаки, по которым модель делает prediction;
- `y` - target, правильный ответ.

Здесь target - `SalePrice`.

В этом baseline notebook мы делаем split 80/20:

- 80% данных для обучения;
- 20% данных для validation.

In [ ]:
# X - все колонки, кроме SalePrice.
X = df_final.drop("SalePrice", axis=1)

# y - только колонка SalePrice, которую хотим предсказывать.
y = df_final["SalePrice"]

# Делим данные на train и validation.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,                 # признаки
    y,                 # target
    train_size=0.8,    # 80% данных идет в train
    test_size=0.2,     # 20% данных идет в validation
    random_state=0     # фиксируем случайность для воспроизводимости
)

# Проверяем размеры получившихся частей.
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

## 9. Обучение моделей

Мы сравним три регрессионные модели:

1. `SVR` - Support Vector Regression.
2. `RandomForestRegressor` - ансамбль деревьев.
3. `LinearRegression` - линейная модель.

Для оценки используем:

- `MAPE`: средняя процентная ошибка. Меньше - лучше.
- `MAE`: средняя абсолютная ошибка в денежных единицах target. Меньше - лучше.
- `RMSE`: корень из средней квадратичной ошибки. Сильнее штрафует большие ошибки. Меньше - лучше.
- `R²`: доля объясненной вариации. Больше - лучше.

In [ ]:
# Создаем словарь моделей: имя -> объект модели.
models = {
    "SVR": SVR(),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=10, random_state=RANDOM_STATE),
    "LinearRegression": LinearRegression(),
}

# Сюда будем складывать метрики каждой модели.
results = []

# Проходим по всем моделям из словаря.
for model_name, model in models.items():
    # fit обучает модель на train данных.
    model.fit(X_train, y_train)
    
    # predict делает предсказания для validation признаков.
    y_pred = model.predict(X_valid)
    
    # MAPE показывает среднюю процентную ошибку.
    mape = mean_absolute_percentage_error(y_valid, y_pred)
    
    # MAE показывает среднюю абсолютную ошибку.
    mae = mean_absolute_error(y_valid, y_pred)
    
    # RMSE считаем как sqrt(MSE).
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    
    # R² показывает, насколько модель лучше простого среднего.
    r2 = r2_score(y_valid, y_pred)
    
    # Добавляем строку результатов в список.
    results.append({
        "model": model_name,
        "MAPE": mape,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

# Превращаем список словарей в DataFrame и сортируем по MAPE.
results_df = pd.DataFrame(results).sort_values("MAPE")

# Показываем таблицу результатов.
results_df

## 10. Визуальная проверка лучшей модели

График `actual vs predicted` показывает, насколько предсказания близки к реальным значениям.

Если модель идеальна, точки лежат на диагональной линии.

In [ ]:
# Берем имя лучшей модели по минимальному MAPE.
best_model_name = results_df.iloc[0]["model"]

# Достаем сам объект лучшей модели из словаря.
best_model = models[best_model_name]

# Получаем prediction лучшей модели на validation set.
best_pred = best_model.predict(X_valid)

# Создаем график.
plt.figure(figsize=(7, 7))

# Каждая точка: x = реальная цена, y = предсказанная цена.
sns.scatterplot(x=y_valid, y=best_pred, alpha=0.6)

# Считаем минимальную и максимальную цену для диагональной линии.
min_price = min(y_valid.min(), best_pred.min())
max_price = max(y_valid.max(), best_pred.max())

# Рисуем идеальную диагональ: actual = predicted.
plt.plot([min_price, max_price], [min_price, max_price], color="red", linestyle="--")

# Подписываем график.
plt.title(f"Actual vs predicted prices: {best_model_name}")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")

# Делаем layout аккуратнее.
plt.tight_layout()

# Показываем график.
plt.show()

## 11. Итог baseline notebook

Что мы сделали:

- загрузили Excel dataset;
- провели базовый EDA;
- удалили `Id`;
- обработали missing values простым способом;
- применили One-Hot Encoding;
- сделали train/validation split 80/20;
- обучили SVR, RandomForestRegressor и LinearRegression;
- сравнили модели по MAPE, MAE, RMSE, R².

Главное ограничение: encoding и cleaning были сделаны до split. Для учебного baseline это похоже на статью, но в более строгом data science pipeline preprocessing нужно обучать только на train. Это исправлено во втором notebook.